# Train PDE-Transformer-TTT on APE/Exponax Data

This notebook shows a minimal Colab/Jupyter workflow for training your own TTT PDE-Transformer model from scratch. It uses `pdetransformer-ttt==0.0.1rc4`, optionally generates APEBench-style data with the package's Exponax simulator, and trains via Lightning without relying on YAML files.

Safe defaults are intentionally small: data generation is off by default and the training dataset list starts with only `diff`. Expand after the first end-to-end run works.

## 1. Install Packages

In [ ]:
%pip install -q pdetransformer-ttt==0.0.1rc4

# Needed only if you want this notebook to generate datasets.
# If data is already available in DATA_DIR, you can skip this cell after installing pdetransformer-ttt.
%pip install -q "jax[cuda12]" exponax==0.1.0 h5py matplotlib vape4d

In [ ]:
import os
import gc
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import torch
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from pdetransformer.core.mixed_channels import PDETransformer, SingleStepSupervised
from pdetransformer.data import MultiDataModule

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('cuda device count:', torch.cuda.device_count() if torch.cuda.is_available() else 0)

## 2. Data Directory and Dataset Lists

`2D_APE_xxl` expects local HDF5 files generated by `pdetransformer.data.simulations_apebench.simulation`, for example `diff.hdf5`, `kolm_flow.hdf5`, and for long-rollout datasets also `kolm_flow_test.hdf5`.

In [ ]:
DATA_DIR = Path('./datasets')
DATA_DIR.mkdir(parents=True, exist_ok=True)

FULL_DATASET_NAMES = [
    'diff', 'hyp', 'burgers', 'kdv', 'ks', 'fisher',
    'gs_alpha', 'gs_beta', 'gs_gamma', 'gs_delta',
    'gs_epsilon', 'gs_theta', 'gs_iota', 'gs_kappa',
    'sh', 'decay_turb', 'kolm_flow',
]

# Full training. For a quick smoke test, swap to ['diff'] or ['diff', 'burgers'].
dataset_names = FULL_DATASET_NAMES

print('cwd:', Path.cwd())
print('DATA_DIR:', DATA_DIR.resolve())
print('training datasets:', dataset_names)

## 3. Optional: Generate Exponax/APE XXL Data

Set `GENERATE_DATA = True` only when you actually want to generate HDF5 files. Full generation is expensive. For a first test, generate only `diff`.

In [ ]:
GENERATE_DATA = False
LOW_RES = True

def run_generation(pde, out_name=None, num_sims=60, test_set=False, low_res=True):
    out_name = out_name or pde
    cmd = [
        'python', '-m', 'pdetransformer.data.simulations_apebench.simulation',
        '--pde', pde,
        '--out_name', out_name,
        '--out_path', str(DATA_DIR),
        '--num_sims', str(num_sims),
    ]
    if test_set:
        cmd.append('--test_set')
    if low_res:
        cmd.append('--low-res')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if GENERATE_DATA:
    # Full generation. Expensive: ~17 datasets, several PDEs need a separate _test file.
    # For a quick smoke test, comment the loops below and run only:
    #     run_generation('diff', num_sims=60, low_res=LOW_RES)

    for pde in ['diff', 'hyp', 'burgers', 'kdv', 'fisher', 'sh']:
        run_generation(pde, num_sims=60, low_res=LOW_RES)

    # ks / decay_turb / kolm_flow: ape_2d_xxl.py reads {pde}_test.hdf5 for the test split.
    for pde in ['ks', 'decay_turb', 'kolm_flow']:
        run_generation(pde, num_sims=60, low_res=LOW_RES)
        run_generation(pde, out_name=f'{pde}_test', num_sims=5, test_set=True, low_res=LOW_RES)

    # gs_alpha/beta/gamma/epsilon: also need a separate _test.hdf5.
    for pde in ['gs_alpha', 'gs_beta', 'gs_gamma', 'gs_epsilon']:
        run_generation(pde, num_sims=10, low_res=LOW_RES)
        run_generation(pde, out_name=f'{pde}_test', num_sims=3, test_set=True, low_res=LOW_RES)

    # gs_delta/theta/iota/kappa: train uses sim0..sim79, test uses sim80..sim99 in the same file.
    for pde in ['gs_delta', 'gs_theta', 'gs_iota', 'gs_kappa']:
        run_generation(pde, num_sims=100, low_res=LOW_RES)
else:
    print('Skipping data generation. Existing files:')
    print(sorted(p.name for p in DATA_DIR.glob('*.hdf5'))[:20])

## 4. Build DataModule

If you use generated Exponax data, keep `dataset_type='2D_APE_xxl'`. If you use HuggingFace scraped APEBench files, use `2D_APE` instead and match its `*-train.hdf5` / `*-test.hdf5` naming convention.

In [ ]:
params_data = {
    'path_index': {'2D_APE_xxl': str(DATA_DIR)},
    'dataset_names': dataset_names,
    'dataset_type': '2D_APE_xxl',
    'unrolling_steps': 1,
    'test_unrolling_steps': 29,
    'batch_size': 8,
    'num_workers': 2,
    'cache_strategy': 'none',
    'different_resolution_strategy': 'none',
    'normalize_data': 'mean-std',
    'normalize_const': 'mean-std',
    # low-res Exponax output is 256x256; model sample_size=128, so downsample 256 -> 128.
    'downsample_factor': 2,
    'max_channels': 2,
}

data_module = MultiDataModule(**params_data)
data_module.setup(stage='fit')
print('DataModule ready')

## 5. Experiment Setup


In [ ]:
MAX_EPOCHS = 100
SEED = 42
NUM_DEVICES = 1
TRAIN_STRATEGY = 'auto'
RUN_ROOT = Path('./ttt_cache_experiments')
RUN_NAME = 'train_once'
RESUME_FROM_CHECKPOINT = False
CHECKPOINT_PATH = './lightning_logs/version_1/checkpoints/epoch=0-step=324.ckpt'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

INFERENCE_CACHE_EVALS = [
    {'name': 'inference_cache_off', 'use_ttt_state_cache_inference': False},
    {'name': 'inference_cache_on', 'use_ttt_state_cache_inference': True},
]

class EpochPrintCallback(L.Callback):
    def on_train_epoch_start(self, trainer, pl_module):
        self._epoch_start_time = time.time()

    def _scalar_metrics(self, trainer, prefixes):
        metrics = {}
        for key, value in trainer.callback_metrics.items():
            if not any(key.startswith(prefix) or prefix in key for prefix in prefixes):
                continue
            if hasattr(value, 'detach') and value.numel() == 1:
                metrics[key] = float(value.detach().cpu())
        return metrics

    def on_train_epoch_end(self, trainer, pl_module):
        elapsed = time.time() - getattr(self, '_epoch_start_time', time.time())
        metrics = self._scalar_metrics(trainer, prefixes=('train', 'loss'))
        metric_text = ', '.join(f'{k}={v:.6g}' for k, v in sorted(metrics.items())) or 'no train metrics yet'
        print(
            f'[train] epoch {trainer.current_epoch + 1}/{trainer.max_epochs} '
            f'global_step={trainer.global_step} elapsed={elapsed/60:.1f} min | {metric_text}'
        )

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        metrics = self._scalar_metrics(trainer, prefixes=('val',))
        metric_text = ', '.join(f'{k}={v:.6g}' for k, v in sorted(metrics.items())) or 'no val metrics yet'
        print(f'[val] epoch {trainer.current_epoch + 1}/{trainer.max_epochs} global_step={trainer.global_step} | {metric_text}')

def build_strategy(use_ttt_state_cache_inference=False):
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = PDETransformer(
        sample_size=128,
        in_channels=2,
        out_channels=2,
        type='PDE-S',
        patch_size=4,
        periodic=True,
        carrier_token_active=False,
        use_ttt_window_attention=True,
        ttt_layer_type='linear',
        ttt_mini_batch_size=16,
        ttt_base_lr=1.0,
    )

    strategy = SingleStepSupervised(
        model=model,
        image_key=0,
        optimizer='adamw',
        use_ttt_state_cache_inference=use_ttt_state_cache_inference,
        use_ttt_state_cache_train=False,
    )
    strategy.learning_rate = 4.0e-5
    return strategy

def build_trainer(run_name):
    checkpoint_callback = ModelCheckpoint(
        dirpath=RUN_ROOT / run_name / 'checkpoints',
        filename='epoch-{epoch:03d}',
        monitor='val/loss_epoch',
        mode='min',
        save_last=True,
        save_top_k=3,
        every_n_epochs=1,
    )
    return L.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=NUM_DEVICES if torch.cuda.is_available() else 1,
        strategy=TRAIN_STRATEGY,
        precision='bf16-mixed' if torch.cuda.is_available() else '32-true',
        accumulate_grad_batches=8,
        callbacks=[checkpoint_callback, EpochPrintCallback()],
        logger=CSVLogger(save_dir=str(RUN_ROOT), name=run_name),
        enable_progress_bar=False,
        log_every_n_steps=10,
    )

probe_strategy = build_strategy(False)
print('Model parameters:', sum(p.numel() for p in probe_strategy.parameters()))
print('Checkpoint directory:', (RUN_ROOT / RUN_NAME / 'checkpoints').resolve())
print('Trainer devices:', NUM_DEVICES if torch.cuda.is_available() else 1, 'strategy:', TRAIN_STRATEGY)
print('Resume from checkpoint:', RESUME_FROM_CHECKPOINT, CHECKPOINT_PATH if RESUME_FROM_CHECKPOINT else '')
del probe_strategy
gc.collect()

## 6. Full Training Test

This trains one model with training-time TTT state cache disabled, matching the YAML default. The trained checkpoint is reused for both inference-cache evaluations.

In [ ]:
data_module.setup(stage='fit')
strategy = build_strategy(use_ttt_state_cache_inference=False)
trainer = build_trainer(RUN_NAME)

ckpt_path = CHECKPOINT_PATH if RESUME_FROM_CHECKPOINT else None
if ckpt_path is not None:
    ckpt_path = str(Path(ckpt_path))
    print('resuming from checkpoint:', ckpt_path)
else:
    print('starting fresh training run')

trainer.fit(strategy, datamodule=data_module, ckpt_path=ckpt_path)
val_metrics = trainer.validate(strategy, datamodule=data_module, verbose=False)

checkpoint_callback = trainer.checkpoint_callback
print('last checkpoint:', checkpoint_callback.last_model_path)
print('best checkpoints:', checkpoint_callback.best_k_models)
print('validation:', val_metrics)

## 7. Compare Inference Cache On/Off

This reuses the same trained weights and runs two autoregressive rollouts: one with inference cache disabled and one with inference cache enabled.

In [ ]:
def run_rollout_check(strategy, datamodule, use_ttt_state_cache_inference, num_frames=29):
    datamodule.setup(stage='test')
    test_loader = datamodule.test_dataloader()
    batch = next(iter(test_loader))

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    strategy.use_ttt_state_cache_inference = use_ttt_state_cache_inference
    strategy.use_ttt_state_cache_train = False
    strategy = strategy.to(device)
    strategy.eval()
    with torch.no_grad():
        prediction, reference = strategy.predict(batch, device=device, num_frames=num_frames)

    prediction_np = np.asarray(prediction)
    reference_np = np.asarray(reference)
    rollout_mse = float(np.mean((prediction_np - reference_np) ** 2))
    return {
        'use_ttt_state_cache_inference': use_ttt_state_cache_inference,
        'use_ttt_state_cache_train': False,
        'prediction_shape': tuple(prediction_np.shape),
        'reference_shape': tuple(reference_np.shape),
        'rollout_mse': rollout_mse,
    }

experiment_results = []
for experiment in INFERENCE_CACHE_EVALS:
    print(f"\n=== Evaluating {experiment['name']} ===")
    result = {
        'name': experiment['name'],
        'checkpoint': trainer.checkpoint_callback.last_model_path,
        'max_epochs': MAX_EPOCHS,
        'val_metrics': val_metrics,
        **run_rollout_check(
            strategy,
            data_module,
            use_ttt_state_cache_inference=experiment['use_ttt_state_cache_inference'],
            num_frames=29,
        ),
    }
    experiment_results.append(result)
    print(json.dumps(result, indent=2))

off = next(r for r in experiment_results if not r['use_ttt_state_cache_inference'])
on = next(r for r in experiment_results if r['use_ttt_state_cache_inference'])

print('cache off rollout mse:', off['rollout_mse'])
print('cache on  rollout mse:', on['rollout_mse'])
print('prediction shapes:', off['prediction_shape'], on['prediction_shape'])

## 8. Check Current Checkpoint Callback


In [ ]:
print(trainer.checkpoint_callback)
print('dirpath:', trainer.checkpoint_callback.dirpath)
print('save_top_k:', trainer.checkpoint_callback.save_top_k)
print('save_last:', trainer.checkpoint_callback.save_last)
print('monitor:', trainer.checkpoint_callback.monitor)
print('best checkpoints:', trainer.checkpoint_callback.best_k_models)
print('last checkpoint:', trainer.checkpoint_callback.last_model_path)

## 9. Find Existing Checkpoint Files


In [ ]:
from pathlib import Path

# Possible checkpoint locations from old and current notebook versions.
candidates = [
    Path('./ttt_cache_experiments/train_once/checkpoints'),
    Path('./ttt_cache_experiments/inference_cache_off/checkpoints'),
    Path('./ttt_cache_experiments/inference_cache_on/checkpoints'),
    Path('./lightning_logs/version_0/checkpoints'),
    Path('./lightning_logs/version_1/checkpoints'),
]

for ckpt_dir in candidates:
    print('\nDIR:', ckpt_dir.resolve())
    if ckpt_dir.exists():
        ckpts = sorted(ckpt_dir.glob('*.ckpt'))
        if ckpts:
            for p in ckpts:
                print(f'  {p.name} | {p.stat().st_size / 1024**2:.1f} MB')
        else:
            print('  exists, but no .ckpt files')
    else:
        print('  not found')

print('\nAll .ckpt under current directory:')
for p in sorted(Path('.').rglob('*.ckpt')):
    print(f'{p} | {p.stat().st_size / 1024**2:.1f} MB')